# CHALLENGE 02 — Rompe tu propio RAG

**Tiempo sugerido:** 35 minutos · **Requiere proveedor configurado** · Trabajo en equipos de 3.

Ingeniería del caos: modifica parámetros extremos para encontrar dónde y por qué falla el sistema.

Encuentra configuraciones donde el RAG:
1. **No encuentra** la información.
2. **Recupera información incorrecta** o contradictoria.
3. **Recupera demasiado contexto** (ruido).
4. **Genera una respuesta incorrecta** con buen contexto.

Para cada falla responde: **¿qué componente causó el problema?**

In [ ]:
# Configuración común: localiza el repositorio, carga .env y el documento de trabajo
import os
import json
from pathlib import Path
from dotenv import load_dotenv

def encontrar_raiz():
    p = Path.cwd().resolve()
    for c in [p, *p.parents]:
        if (c / "data").is_dir() and (c / "pyproject.toml").exists():
            return c
    raise FileNotFoundError("No encontré la carpeta data/. Abre el notebook desde el repositorio del curso.")

ROOT = encontrar_raiz()
load_dotenv(ROOT / ".env")
DATA_DIR = ROOT / "data"

DOCUMENTOS = {
    "cafeteria":   ("cafeteria_nebula_manual.md",      "cafeteria_preguntas.json"),
    "streamly":    ("streamly_api_docs.md",            "streamly_preguntas.json"),
    "veterinaria": ("clinica_veterinaria_huellitas.md", "veterinaria_preguntas.json"),
    "mi_documento": ("mi_documento.md",                 None),  # genera el tuyo con los prompts de alumnos/02_material_del_alumno.md
}

def cargar_documento(nombre):
    archivo, _ = DOCUMENTOS[nombre]
    return (DATA_DIR / archivo).read_text(encoding="utf-8")

def cargar_preguntas(nombre):
    _, archivo = DOCUMENTOS[nombre]
    if archivo is None:
        return []
    return json.loads((DATA_DIR / archivo).read_text(encoding="utf-8"))

print("Repositorio:", ROOT)

In [ ]:
# 👉 Elige el documento con el que vas a trabajar
DOCUMENTO = "cafeteria"   # "cafeteria" | "streamly" | "veterinaria" | "mi_documento"

texto = cargar_documento(DOCUMENTO)
print(f"Documento: {DOCUMENTO} — {len(texto):,} caracteres, {len(texto.split()):,} palabras")
print(texto[:500], "...")

In [ ]:
# Funciones de chunking (las construiste en el LAB 02)
import re

def chunk_text(text, chunk_size=500, overlap=100):
    if overlap >= chunk_size:
        raise ValueError("overlap debe ser menor que chunk_size")
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + chunk_size])
        start += chunk_size - overlap
    return chunks

def chunk_by_section(text, max_chars=1200):
    """Divide por encabezados Markdown y subdivide secciones largas por párrafos."""
    resultado, seccion, buffer = [], "Inicio", []

    def vaciar():
        cuerpo = "\n".join(buffer).strip()
        if not cuerpo:
            return
        parrafos, actual = [p for p in cuerpo.split("\n\n") if p.strip()], ""
        for p in parrafos:
            if actual and len(actual) + len(p) > max_chars:
                resultado.append({"seccion": seccion, "texto": f"{seccion}\n{actual.strip()}"})
                actual = ""
            actual += p + "\n\n"
        if actual.strip():
            resultado.append({"seccion": seccion, "texto": f"{seccion}\n{actual.strip()}"})

    for linea in text.splitlines():
        m = re.match(r"^#{1,6}\s+(.*)", linea)
        if m:
            vaciar()
            seccion, buffer = m.group(1).strip(), []
        else:
            buffer.append(linea)
    vaciar()
    return resultado

In [ ]:
# Proveedor híbrido: OpenAI (API) u Ollama (local). Se configura en .env con LLM_PROVEEDOR.
# Ollama expone un endpoint compatible con la API de OpenAI, así que usamos el mismo cliente.
import numpy as np
from openai import OpenAI

PROVEEDOR = os.getenv("LLM_PROVEEDOR", "openai").lower()

if PROVEEDOR == "openai":
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Falta OPENAI_API_KEY en .env (o cambia LLM_PROVEEDOR=ollama).")
    client = OpenAI()
    EMBEDDING_MODEL = "text-embedding-3-small"
    CHAT_MODEL = "gpt-4.1-mini"
elif PROVEEDOR == "ollama":
    client = OpenAI(base_url=os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"), api_key="ollama")
    EMBEDDING_MODEL = os.getenv("OLLAMA_EMBEDDING_MODEL", "nomic-embed-text")
    CHAT_MODEL = os.getenv("OLLAMA_CHAT_MODEL", "llama3.2")
else:
    raise ValueError("LLM_PROVEEDOR debe ser 'openai' u 'ollama'")

# Nota: los nombres de modelos cambian con el tiempo; verifica la documentación vigente del proveedor.
print(f"Proveedor: {PROVEEDOR} | embeddings: {EMBEDDING_MODEL} | chat: {CHAT_MODEL}")

_cache_embeddings = {}

def get_embeddings(textos, batch_size=64):
    """Devuelve una matriz (n_textos, dimensiones). Cachea resultados para no pagar dos veces."""
    faltantes = [t for t in dict.fromkeys(textos) if t not in _cache_embeddings]
    for i in range(0, len(faltantes), batch_size):
        lote = faltantes[i:i + batch_size]
        resp = client.embeddings.create(model=EMBEDDING_MODEL, input=lote)
        for t, item in zip(lote, resp.data):
            _cache_embeddings[t] = item.embedding
    return np.array([_cache_embeddings[t] for t in textos], dtype=float)

## 1. Pipeline parametrizable (ya hecho)

Todo el RAG depende de un diccionario `config`. Tu trabajo es **cambiar la configuración y medir**.

In [ ]:
import pandas as pd

PROMPTS = {
    "estricto": ("Responde SOLO con el contexto. Cita los ids como [n]. "
                 "Si no está en el contexto responde: 'No encuentro esa información en los documentos.' "
                 "Si hay datos contradictorios, usa el más reciente."),
    "laxo": "Eres un asistente útil. Responde la pregunta del usuario.",
}

CONFIG_BASE = {
    "estrategia": "secciones",   # "secciones" | "caracteres"
    "chunk_size": 1200,
    "overlap": 0,
    "k": 3,
    "prompt": "estricto",        # "estricto" | "laxo"
    "documentos": [DOCUMENTO],
}

def indexar(config):
    textos, metas = [], []
    for doc in config["documentos"]:
        t = cargar_documento(doc)
        if config["estrategia"] == "secciones":
            piezas = [c["texto"] for c in chunk_by_section(t, max_chars=config["chunk_size"])]
        else:
            piezas = chunk_text(t, config["chunk_size"], config["overlap"])
        textos += piezas
        metas += [{"source": doc} for _ in piezas]
    return textos, metas, get_embeddings(textos)

def recuperar(pregunta, indice, k):
    textos, metas, M = indice
    q = get_embeddings([pregunta])[0]
    scores = (M @ q) / (np.linalg.norm(M, axis=1) * np.linalg.norm(q))
    top = np.argsort(scores)[::-1][:k]
    return [{"id": int(i), "score": float(scores[i]), "texto": textos[i]} for i in top]

def responder(pregunta, recuperados, config):
    contexto = "\n\n".join(f"[{r['id']}]\n{r['texto']}" for r in recuperados)
    mensajes = [{"role": "system", "content": PROMPTS[config["prompt"]]},
                {"role": "user", "content": f"CONTEXTO:\n{contexto}\n\nPREGUNTA: {pregunta}"}]
    r = client.chat.completions.create(model=CHAT_MODEL, messages=mensajes, temperature=0)
    return r.choices[0].message.content

## 2. Evaluación del retrieval (antes que la respuesta)

Cada pregunta de `data/<documento>_preguntas.json` incluye una **evidencia**: un fragmento de texto exacto que debe
aparecer en algún chunk recuperado. Con eso medimos:

- **hit@k**: ¿la evidencia aparece en el Top-K? (1 = sí, 0 = no)
- **tokens de contexto**: cuánto texto enviamos al LLM.

### ✏️ Ejercicio 1 — completa `hit`

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("o200k_base")

def hit(recuperados, evidencias):
    """1 si alguna evidencia aparece completa en algún chunk recuperado, si no 0."""
    # TODO
    ...

In [ ]:
# ✅ Autoverificación
assert hit([{"texto": "abc def"}], ["def"]) == 1
assert hit([{"texto": "abc de"}, {"texto": "f ghi"}], ["def"]) == 0
print("✅ Ejercicio 1 correcto")

In [ ]:
PREGUNTAS = cargar_preguntas(DOCUMENTO)
assert PREGUNTAS, "Este challenge necesita un archivo de preguntas: usa cafeteria, streamly o veterinaria."

def evaluar_retrieval(config, nombre="experimento"):
    indice = indexar(config)
    filas = []
    for p in PREGUNTAS:
        if not p["evidencia"]:
            continue
        rec = recuperar(p["pregunta"], indice, config["k"])
        filas.append({"id": p["id"], "tipo": p["tipo"], "hit": hit(rec, p["evidencia"]),
                      "tokens_contexto": sum(len(enc.encode(r["texto"])) for r in rec)})
    df = pd.DataFrame(filas)
    resumen = {"experimento": nombre, "n_chunks": len(indice[0]), "hit_rate": round(df["hit"].mean(), 2),
               "tokens_contexto_prom": int(df["tokens_contexto"].mean()), **{k: v for k, v in config.items() if k != "documentos"}}
    return resumen, df

resumen, detalle = evaluar_retrieval(CONFIG_BASE, "base")
print(resumen)
detalle

## 3. Experimentos

### ✏️ Ejercicio 2 — rompe el RAG

Agrega configuraciones a `EXPERIMENTOS` (ya hay algunas sugeridas) y compara `hit_rate` y `tokens_contexto_prom`.
Ideas: chunk diminuto, chunk gigante, overlap enorme, K = 1, K = 15.

In [ ]:
EXPERIMENTOS = {
    "base": CONFIG_BASE,
    "chunk diminuto": {**CONFIG_BASE, "estrategia": "caracteres", "chunk_size": 60, "overlap": 0},
    "K = 1": {**CONFIG_BASE, "k": 1},
    # TODO: agrega al menos 3 experimentos más
}

resultados = pd.DataFrame([evaluar_retrieval(cfg, nombre)[0] for nombre, cfg in EXPERIMENTOS.items()])
resultados

**Interpreta:** un `hit_rate` alto con muchísimos `tokens_contexto` no es una victoria: estás mandando "todo el documento".
Buscamos **alto hit_rate con pocos tokens**.

## 4. Rompe la generación

Ahora el retrieval puede estar bien, pero la respuesta no. Prueba:
- Prompt `laxo` con preguntas **fuera** del documento → ¿inventa?
- Preguntas de **conflicto** (datos viejos vs. nuevos) con K pequeño → ¿elige el dato vigente?
- Agregar **otros documentos** al índice → ¿se mezclan fuentes? (p. ej. "¿cuál es el horario?")

In [ ]:
def probar(pregunta, config):
    indice = indexar(config)
    rec = recuperar(pregunta, indice, config["k"])
    print(f"❓ {pregunta}  | prompt={config['prompt']} k={config['k']} docs={config['documentos']}")
    print("📚", [r["texto"].splitlines()[0][:40] for r in rec])
    print("💬", responder(pregunta, rec, config), "\n")

fuera = [p for p in PREGUNTAS if p["tipo"] == "fuera"][0]["pregunta"]
conflicto = [p for p in PREGUNTAS if p["tipo"] == "conflicto"][0]["pregunta"]

probar(fuera, CONFIG_BASE)
probar(fuera, {**CONFIG_BASE, "prompt": "laxo"})
probar(conflicto, {**CONFIG_BASE, "k": 1})
probar("¿Cuál es el horario de atención?", {**CONFIG_BASE, "documentos": ["cafeteria", "streamly", "veterinaria"]})

## 5. Bitácora de fallas

### ✏️ Ejercicio 3 — Diagnóstico

Llena una fila por cada falla que encontraste. Componentes posibles:
`documentos`, `chunking`, `embeddings`, `retrieval (K)`, `contexto`, `prompt`, `LLM`.

In [ ]:
bitacora = pd.DataFrame([
    # TODO: {"falla": "...", "configuracion": "...", "componente_causante": "...", "como_lo_arreglarias": "..."},
])
bitacora

## Pregunta final

**Si mañana tu RAG produce respuestas incorrectas, ¿qué componente investigarías primero y por qué?**

> ...